Credits: this notebook belongs to [Practical DL](https://docs.google.com/forms/d/e/1FAIpQLScvrVtuwrHSlxWqHnLt1V-_7h2eON_mlRR6MUb3xEe5x9LuoA/viewform?usp=sf_link) course by Yandex School of Data Analysis.

In [16]:
import numpy as np

**Module** is an abstract class which defines fundamental methods necessary for a training a neural network. You do not need to change anything here, just read the comments.

In [17]:
class Module(object):
    """
    Basically, you can think of a module as of a something (black box) 
    which can process `input` data and produce `ouput` data.
    This is like applying a function which is called `forward`: 

        output = module.forward(input)

    The module should be able to perform a backward pass: to differentiate the `forward` function. 
    More, it should be able to differentiate it if is a part of chain (chain rule).
    The latter implies there is a gradient from previous step of a chain rule. 

        gradInput = module.backward(input, gradOutput)
    """

    def __init__(self):
        self.output = None
        self.gradInput = None
        self.training = True

    def forward(self, input):
        """
        Takes an input object, and computes the corresponding output of the module.
        """
        return self.updateOutput(input)

    def backward(self, input, gradOutput):
        """
        Performs a backpropagation step through the module, with respect to the given input.

        This includes 
         - computing a gradient w.r.t. `input` (is needed for further backprop),
         - computing a gradient w.r.t. parameters (to update parameters while optimizing).
        """
        self.updateGradInput(input, gradOutput)
        self.accGradParameters(input, gradOutput)
        return self.gradInput

    def updateOutput(self, input):
        """
        Computes the output using the current parameter set of the class and input.
        This function returns the result which is stored in the `output` field.

        Make sure to both store the data in `output` field and return it. 
        """

        # The easiest case:

        # self.output = input
        # return self.output

        pass

    def updateGradInput(self, input, gradOutput):
        """
        Computing the gradient of the module with respect to its own input. 
        This is returned in `gradInput`. Also, the `gradInput` state variable is updated accordingly.

        The shape of `gradInput` is always the same as the shape of `input`.

        Make sure to both store the gradients in `gradInput` field and return it.
        """

        # The easiest case:

        # self.gradInput = gradOutput
        # return self.gradInput

        pass

    def accGradParameters(self, input, gradOutput):
        """
        Computing the gradient of the module with respect to its own parameters.
        No need to override if module has no parameters (e.g. ReLU).
        """
        pass

    def zeroGradParameters(self):
        """
        Zeroes `gradParams` variable if the module has params.
        """
        pass

    def getParameters(self):
        """
        Returns a list with its parameters. 
        If the module does not have parameters return empty list. 
        """
        return []

    def getGradParameters(self):
        """
        Returns a list with gradients with respect to its parameters. 
        If the module does not have parameters return empty list. 
        """
        return []

    def train(self):
        """
        Sets training mode for the module.
        Training and testing behaviour differs for Dropout, BatchNorm.
        """
        self.training = True

    def evaluate(self):
        """
        Sets evaluation mode for the module.
        Training and testing behaviour differs for Dropout, BatchNorm.
        """
        self.training = False

    def __repr__(self):
        """
        Pretty printing. Should be overrided in every module if you want 
        to have readable description. 
        """
        return "Module"

# Sequential container

**Define** a forward and backward pass procedures.

In [ ]:
class Sequential(Module):
    """
    Последовательно применяет под-модули к входу.
    Интерфейс полностью совместим с тестом `test_Sequential`.
    """

    def __init__(self):
        super().__init__()
        self.modules: list[Module] = []
        self.training = True
        self._intermediate = None       # буфер для входов каждого модуля

    # --------------------------------------------------------------------- #
    #  СТРОЕНИЕ СЕТИ
    # --------------------------------------------------------------------- #
    def add(self, module: "Module") -> None:
        self.modules.append(module)

    # --------------------------------------------------------------------- #
    #  FORWARD
    # --------------------------------------------------------------------- #
    def updateOutput(self, x):
        """
        y_0 = modules[0](x)
        y_1 = modules[1](y_0)
        ...
        y_n = modules[n](y_{n-1})
        """
        self._intermediate = [x]            # x ≡ y_{-1}
        out = x
        for m in self.modules:
            out = m.updateOutput(out)
            self._intermediate.append(out)  # y_k

        self.output = out
        return self.output

    # --------------------------------------------------------------------- #
    #  BACKWARD
    # --------------------------------------------------------------------- #
    def backward(self, x, gradOutput):
        """
        g_{k-1} = modules[k-1].backward(y_{k-2}, g_k)
        Здесь g_k — градиент по выходу k-го модуля.
        Для accGradParameters нужно передавать именно g_k,
        а не градиент по входу модуля.
        """
        grad_out = gradOutput                     # g_n
        for idx in range(len(self.modules) - 1, -1, -1):
            module = self.modules[idx]

            # вход текущего модуля ‑ это выход предыдущего
            module_input = self._intermediate[idx]

            # backward возвращает градиент по входу модуля (g_{k-1})
            grad_in = module.backward(module_input, grad_out)

            # аккумулируем градиенты параметров, если они есть
            if hasattr(module, "accGradParameters"):
                module.accGradParameters(module_input, grad_out)

            # переходим к предыдущему модулю
            grad_out = grad_in

        self.gradInput = grad_out                # градиент по x
        return self.gradInput

    # --------------------------------------------------------------------- #
    #  СБРОС ГРАДИЕНТОВ
    # --------------------------------------------------------------------- #
    def zeroGradParameters(self):
        for m in self.modules:
            if hasattr(m, "zeroGradParameters"):
                m.zeroGradParameters()

    # --------------------------------------------------------------------- #
    #  ДОСТУП К ПАРАМЕТРАМ
    # --------------------------------------------------------------------- #
    def getParameters(self):
        return [m.getParameters() for m in self.modules
                if hasattr(m, "getParameters")]

    def getGradParameters(self):
        return [m.getGradParameters() for m in self.modules
                if hasattr(m, "getGradParameters")]

    # --------------------------------------------------------------------- #
    #  TRAIN / EVAL
    # --------------------------------------------------------------------- #
    def train(self) -> None:
        self.training = True
        for m in self.modules:
            if hasattr(m, "train"):
                m.train()

    def evaluate(self) -> None:
        self.training = False
        for m in self.modules:
            if hasattr(m, "evaluate"):
                m.evaluate()

    # --------------------------------------------------------------------- #
    #  ПРОЧЕЕ
    # --------------------------------------------------------------------- #
    def __getitem__(self, idx):
        return self.modules[idx]

    def __repr__(self):
        return "".join(f"{m}\n" for m in self.modules)

# Layers

## 1. Linear transform layer
Also known as dense layer, fully-connected layer, FC-layer, InnerProductLayer (in caffe), affine transform
- input:   **`batch_size x n_feats1`**
- output: **`batch_size x n_feats2`**

In [ ]:
class Linear(Module):
    """
    A module which applies a linear transformation 
    A common name is fully-connected layer, InnerProductLayer in caffe. 

    The module should work with 2D input of shape (n_samples, n_feature).
    """

    def __init__(self, n_in, n_out):
        super(Linear, self).__init__()

        # This is a nice initialization
        stdv = 1. / np.sqrt(n_in)
        self.W = np.random.uniform(-stdv, stdv, size=(n_out, n_in))
        self.b = np.random.uniform(-stdv, stdv, size=n_out)

        self.gradW = np.zeros_like(self.W)
        self.gradb = np.zeros_like(self.b)

    def updateOutput(self, input):
        self.output = np.dot(input, self.W.T) + self.b
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = np.dot(gradOutput, self.W)
        return self.gradInput

    def accGradParameters(self, input, gradOutput):
        self.gradW += np.dot(gradOutput.T, input)
        self.gradb += np.sum(gradOutput, axis=0)

    def zeroGradParameters(self):
        self.gradW.fill(0)
        self.gradb.fill(0)

    def getParameters(self):
        return [self.W, self.b]

    def getGradParameters(self):
        return [self.gradW, self.gradb]

    def __repr__(self):
        s = self.W.shape
        q = 'Linear %d -> %d' % (s[1], s[0])
        return q

## 2. SoftMax
- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

$\text{softmax}(x)_i = \frac{\exp x_i} {\sum_j \exp x_j}$

Recall that $\text{softmax}(x) == \text{softmax}(x - \text{const})$. It makes possible to avoid computing exp() from large argument.

In [ ]:
class SoftMax(Module):
    def __init__(self):
        super(SoftMax, self).__init__()

    def updateOutput(self, input):
        shifted = input - np.max(input, axis=1, keepdims=True)
        exp_shifted = np.exp(shifted)
        sum_exp = np.sum(exp_shifted, axis=1, keepdims=True)
        self.output = exp_shifted / sum_exp
        return self.output

    def updateGradInput(self, input, gradOutput):
        dotted = np.sum(self.output * gradOutput, axis=1, keepdims=True)
        self.gradInput = self.output * (gradOutput - dotted)
        return self.gradInput

    def __repr__(self):
        return "SoftMax"

## 3. LogSoftMax
- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

$\text{logsoftmax}(x)_i = \log\text{softmax}(x)_i = x_i - \log {\sum_j \exp x_j}$

The main goal of this layer is to be used in computation of log-likelihood loss.

In [ ]:
class LogSoftMax(Module):
    def __init__(self):
        super(LogSoftMax, self).__init__()
        self.training = True
        self.eps = 1e-10

    def train(self):
        self.training = True

    def evaluate(self):
        self.training = False

    def updateOutput(self, input):
        max_per_row = np.max(input, axis=1, keepdims=True)
        shifted = input - max_per_row

        exp_shifted = np.exp(shifted)
        sum_exp = np.sum(exp_shifted, axis=1, keepdims=True) + self.eps

        self.output = shifted - np.log(sum_exp)
        return self.output

    def updateGradInput(self, input, gradOutput):
        softmax = np.exp(self.output)
        softmax = np.clip(softmax, self.eps, 1.0)

        sum_grad = np.sum(gradOutput, axis=1, keepdims=True)

        self.gradInput = gradOutput - (softmax * sum_grad)
        return self.gradInput

    def __repr__(self):
        return "LogSoftMax"

## 4. Batch normalization
One of the most significant recent ideas that impacted NNs a lot is [**Batch normalization**](http://arxiv.org/abs/1502.03167). The idea is simple, yet effective: the features should be whitened ($mean = 0$, $std = 1$) all the way through NN. This improves the convergence for deep models letting it train them for days but not weeks. **You are** to implement the first part of the layer: features normalization. The second part (`ChannelwiseScaling` layer) is implemented below.

- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

The layer should work as follows. While training (`self.training == True`) it transforms input as $$y = \frac{x - \mu}  {\sqrt{\sigma + \epsilon}}$$
where $\mu$ and $\sigma$ - mean and variance of feature values in **batch** and $\epsilon$ is just a small number for numericall stability. Also during training, layer should maintain exponential moving average values for mean and variance: 
```
    self.moving_mean = self.moving_mean * alpha + batch_mean * (1 - alpha)
    self.moving_variance = self.moving_variance * alpha + batch_variance * (1 - alpha)
```
During testing (`self.training == False`) the layer normalizes input using moving_mean and moving_variance. 

Note that decomposition of batch normalization on normalization itself and channelwise scaling here is just a common **implementation** choice. In general "batch normalization" always assumes normalization + scaling.

In [ ]:
class BatchNormalization(Module):

    EPS = 1e-3

    def __init__(self, alpha: float = 0.0):
        super().__init__()
        self.alpha = alpha
        self.moving_mean = None
        self.moving_variance = None
        self.training = True

    def train(self):
        self.training = True

    def evaluate(self):
        self.training = False

    def updateOutput(self, x: np.ndarray) -> np.ndarray:
        N = x.shape[0]

        if self.training:
            batch_mean = x.mean(axis=0)
            centered = x - batch_mean
            var_biased = (centered ** 2).mean(axis=0)

            self.output = centered / np.sqrt(var_biased + self.EPS)

            if self.moving_mean is None:
                self.moving_mean = batch_mean.copy()
                self.moving_variance = (var_biased * N / (N - 1)
                                        if N > 1 else np.zeros_like(var_biased))
            else:
                self.moving_mean = (self.alpha * self.moving_mean +
                                    (1 - self.alpha) * batch_mean)

                unbiased_var = (var_biased * N / (N - 1)
                                if N > 1 else np.zeros_like(var_biased))
                self.moving_variance = (self.alpha * self.moving_variance +
                                        (1 - self.alpha) * unbiased_var)
        else:
            self.output = (x - self.moving_mean) / \
                np.sqrt(self.moving_variance + self.EPS)

        return self.output

    def updateGradInput(self, x: np.ndarray, grad_out: np.ndarray) -> np.ndarray:

        N = x.shape[0]

        if not self.training or N == 0:
            self.gradInput = grad_out / \
                np.sqrt(self.moving_variance + self.EPS)
            return self.gradInput

        mu = x.mean(axis=0)
        var = ((x - mu) ** 2).mean(axis=0)
        inv_std = 1.0 / np.sqrt(var + self.EPS)

        grad_sum = grad_out.sum(axis=0)
        dot = (grad_out * (x - mu)).sum(axis=0)

        self.gradInput = (grad_out * N
                          - grad_sum
                          - (x - mu) * dot / (var + self.EPS))
        self.gradInput *= inv_std / N

        return self.gradInput

    def __repr__(self):
        return "BatchNormalization"

In [23]:
class ChannelwiseScaling(Module):
    """
       Implements linear transform of input y = \gamma * x + \beta
       where \gamma, \beta - learnable vectors of length x.shape[-1]
    """

    def __init__(self, n_out):
        super(ChannelwiseScaling, self).__init__()

        stdv = 1./np.sqrt(n_out)
        self.gamma = np.random.uniform(-stdv, stdv, size=n_out)
        self.beta = np.random.uniform(-stdv, stdv, size=n_out)

        self.gradGamma = np.zeros_like(self.gamma)
        self.gradBeta = np.zeros_like(self.beta)

    def updateOutput(self, input):
        self.output = input * self.gamma + self.beta
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput * self.gamma
        return self.gradInput

    def accGradParameters(self, input, gradOutput):
        self.gradBeta = np.sum(gradOutput, axis=0)
        self.gradGamma = np.sum(gradOutput*input, axis=0)

    def zeroGradParameters(self):
        self.gradGamma.fill(0)
        self.gradBeta.fill(0)

    def getParameters(self):
        return [self.gamma, self.beta]

    def getGradParameters(self):
        return [self.gradGamma, self.gradBeta]

    def __repr__(self):
        return "ChannelwiseScaling"

<>:3: SyntaxWarning: invalid escape sequence '\g'
<>:3: SyntaxWarning: invalid escape sequence '\g'
/var/folders/66/ylwdy6t93213rp9v263z67tm0000gn/T/ipykernel_1186/536005684.py:3: SyntaxWarning: invalid escape sequence '\g'
  Implements linear transform of input y = \gamma * x + \beta


Practical notes. If BatchNormalization is placed after a linear transformation layer (including dense layer, convolutions, channelwise scaling) that implements function like `y = weight * x + bias`, than bias adding become useless and could be omitted since its effect will be discarded while batch mean subtraction. If BatchNormalization (followed by `ChannelwiseScaling`) is placed before a layer that propagates scale (including ReLU, LeakyReLU) followed by any linear transformation layer than parameter `gamma` in `ChannelwiseScaling` could be freezed since it could be absorbed into the linear transformation layer.

## 5. Dropout
Implement [**dropout**](https://www.cs.toronto.edu/~hinton/absps/JMLRdropout.pdf). The idea and implementation is really simple: just multimply the input by $Bernoulli(p)$ mask. Here $p$ is probability of an element to be zeroed.

This has proven to be an effective technique for regularization and preventing the co-adaptation of neurons.

While training (`self.training == True`) it should sample a mask on each iteration (for every batch), zero out elements and multiply elements by $1 / (1 - p)$. The latter is needed for keeping mean values of features close to mean values which will be in test mode. When testing this module should implement identity transform i.e. `self.output = input`.

- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

In [ ]:
class Dropout(Module):
    def __init__(self, p=0.5):
        super(Dropout, self).__init__()
        self.p = p
        self.mask = None
        self.training = True

    def train(self):
        self.training = True

    def evaluate(self):
        self.training = False

    def updateOutput(self, input):
        if self.training:
            self.mask = np.random.binomial(1, 1 - self.p, size=input.shape)
            self.output = (input * self.mask) / (1 - self.p)
        else:
            self.output = input
        return self.output

    def updateGradInput(self, input, gradOutput):
        if self.training:
            self.gradInput = (gradOutput * self.mask) / (1 - self.p)
        else:
            self.gradInput = gradOutput
        return self.gradInput

    def __repr__(self):
        return "Dropout"

# Activation functions

Here's the complete example for the **Rectified Linear Unit** non-linearity (aka **ReLU**): 

In [25]:
class ReLU(Module):
    def __init__(self):
        super(ReLU, self).__init__()

    def updateOutput(self, input):
        self.output = np.maximum(input, 0)
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = np.multiply(gradOutput, input > 0)
        return self.gradInput

    def __repr__(self):
        return "ReLU"

## 6. Leaky ReLU
Implement [**Leaky Rectified Linear Unit**](http://en.wikipedia.org/wiki%2FRectifier_%28neural_networks%29%23Leaky_ReLUs). Expriment with slope. 

In [ ]:
class LeakyReLU(Module):
    def __init__(self, slope=0.03):
        super(LeakyReLU, self).__init__()
        self.slope = slope

    def updateOutput(self, input):
        self.output = np.maximum(0, input) + self.slope * np.minimum(0, input)
        return self.output

    def updateGradInput(self, input, gradOutput):
        grad = np.ones_like(input)
        grad[input <= 0] = self.slope
        self.gradInput = gradOutput * grad
        return self.gradInput

    def __repr__(self):
        return "LeakyReLU"

## 7. ELU
Implement [**Exponential Linear Units**](http://arxiv.org/abs/1511.07289) activations.

In [ ]:
class ELU(Module):
    def __init__(self, alpha=1.0):
        super(ELU, self).__init__()
        self.alpha = alpha

    def updateOutput(self, input):
        self.output = np.maximum(0, input) + self.alpha * \
            (np.minimum(0, np.exp(input) - 1))
        return self.output

    def updateGradInput(self, input, gradOutput):
        grad = np.ones_like(input)
        grad[input <= 0] = self.alpha * np.exp(input[input <= 0])
        self.gradInput = gradOutput * grad
        return self.gradInput

    def __repr__(self):
        return "ELU"

## 8. SoftPlus
Implement [**SoftPlus**](https://en.wikipedia.org/wiki%2FRectifier_%28neural_networks%29) activations. Look, how they look a lot like ReLU.

In [28]:
class SoftPlus(Module):
    def __init__(self):
        super(SoftPlus, self).__init__()

    def updateOutput(self, input):
        self.output = np.log(np.exp(input) + 1)
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = (1 / (1 + np.exp(-input))) * gradOutput
        return self.gradInput

    def __repr__(self):
        return "SoftPlus"

# Criterions

Criterions are used to score the models answers. 

In [29]:
class Criterion(object):
    def __init__(self):
        self.output = None
        self.gradInput = None

    def forward(self, input, target):
        """
            Given an input and a target, compute the loss function 
            associated to the criterion and return the result.

            For consistency this function should not be overrided,
            all the code goes in `updateOutput`.
        """
        return self.updateOutput(input, target)

    def backward(self, input, target):
        """
            Given an input and a target, compute the gradients of the loss function
            associated to the criterion and return the result. 

            For consistency this function should not be overrided,
            all the code goes in `updateGradInput`.
        """
        return self.updateGradInput(input, target)

    def updateOutput(self, input, target):
        """
        Function to override.
        """
        return self.output

    def updateGradInput(self, input, target):
        """
        Function to override.
        """
        return self.gradInput

    def __repr__(self):
        """
        Pretty printing. Should be overrided in every module if you want 
        to have readable description. 
        """
        return "Criterion"

The **MSECriterion**, which is basic L2 norm usually used for regression, is implemented here for you.
- input:   **`batch_size x n_feats`**
- target: **`batch_size x n_feats`**
- output: **scalar**

In [30]:
class MSECriterion(Criterion):
    def __init__(self):
        super(MSECriterion, self).__init__()

    def updateOutput(self, input, target):
        self.output = np.sum(np.power(input - target, 2)) / input.shape[0]
        return self.output

    def updateGradInput(self, input, target):
        self.gradInput = (input - target) * 2 / input.shape[0]
        return self.gradInput

    def __repr__(self):
        return "MSECriterion"

## 9. Negative LogLikelihood criterion (numerically unstable)
You task is to implement the **ClassNLLCriterion**. It should implement [multiclass log loss](http://scikit-learn.org/stable/modules/model_evaluation.html#log-loss). Nevertheless there is a sum over `y` (target) in that formula, 
remember that targets are one-hot encoded. This fact simplifies the computations a lot. Note, that criterions are the only places, where you divide by batch size. Also there is a small hack with adding small number to probabilities to avoid computing log(0).
- input:   **`batch_size x n_feats`** - probabilities
- target: **`batch_size x n_feats`** - one-hot representation of ground truth
- output: **scalar**



In [31]:
class ClassNLLCriterionUnstable(Criterion):
    EPS = 1e-15

    def __init__(self):
        a = super(ClassNLLCriterionUnstable, self)
        super(ClassNLLCriterionUnstable, self).__init__()

    def updateOutput(self, input, target):

        # Use this trick to avoid numerical errors
        input_clamp = np.clip(input, self.EPS, 1 - self.EPS)
        self.output = -np.sum(target * np.log(input_clamp)) / input.shape[0]
        return self.output

    def updateGradInput(self, input, target):

        # Use this trick to avoid numerical errors
        input_clamp = np.clip(input, self.EPS, 1 - self.EPS)
        self.gradInput = -(target / input) / input.shape[0]
        return self.gradInput

    def __repr__(self):
        return "ClassNLLCriterionUnstable"

## 10. Negative LogLikelihood criterion (numerically stable)
- input:   **`batch_size x n_feats`** - log probabilities
- target: **`batch_size x n_feats`** - one-hot representation of ground truth
- output: **scalar**

Task is similar to the previous one, but now the criterion input is the output of log-softmax layer. This decomposition allows us to avoid problems with computation of forward and backward of log().

In [32]:
class ClassNLLCriterion(Criterion):
    def __init__(self):
        a = super(ClassNLLCriterion, self)
        super(ClassNLLCriterion, self).__init__()

    def updateOutput(self, input, target):
        self.output = -np.sum(target * input) / input.shape[0]
        return self.output

    def updateGradInput(self, input, target):
        self.gradInput = -target / input.shape[0]
        return self.gradInput

    def __repr__(self):
        return "ClassNLLCriterion"

# Optimizers

### SGD optimizer with momentum
- `variables` - list of lists of variables (one list per layer)
- `gradients` - list of lists of current gradients (same structure as for `variables`, one array for each var)
- `config` - dict with optimization parameters (`learning_rate` and `momentum`)
- `state` - dict with optimizator state (used to save accumulated gradients)

In [33]:
def sgd_momentum(variables, gradients, config, state):
    # 'variables' and 'gradients' have complex structure, accumulated_grads will be stored in a simpler one
    state.setdefault('accumulated_grads', {})

    var_index = 0
    for current_layer_vars, current_layer_grads in zip(variables, gradients):
        for current_var, current_grad in zip(current_layer_vars, current_layer_grads):

            old_grad = state['accumulated_grads'].setdefault(
                var_index, np.zeros_like(current_grad))

            np.add(config['momentum'] * old_grad,
                   config['learning_rate'] * current_grad, out=old_grad)

            current_var -= old_grad
            var_index += 1

## 11. [Adam](https://arxiv.org/pdf/1412.6980.pdf) optimizer
- `variables` - list of lists of variables (one list per layer)
- `gradients` - list of lists of current gradients (same structure as for `variables`, one array for each var)
- `config` - dict with optimization parameters (`learning_rate`, `beta1`, `beta2`, `epsilon`)
- `state` - dict with optimizator state (used to save 1st and 2nd moment for vars)

Formulas for optimizer:

Current step learning rate: $$\text{lr}_t = \text{learning_rate} * \frac{\sqrt{1-\beta_2^t}} {1-\beta_1^t}$$
First moment of var: $$\mu_t = \beta_1 * \mu_{t-1} + (1 - \beta_1)*g$$ 
Second moment of var: $$v_t = \beta_2 * v_{t-1} + (1 - \beta_2)*g*g$$
New values of var: $$\text{variable} = \text{variable} - \text{lr}_t * \frac{m_t}{\sqrt{v_t} + \epsilon}$$

In [ ]:
def adam_optimizer(variables, gradients, config, state):

    state.setdefault('m', {})      # 1-е моменты
    state.setdefault('v', {})      # 2-е моменты
    state.setdefault('t', 0)       # номер шага
    state['t'] += 1                # следующий шаг

    lr = config['learning_rate']
    beta1 = config['beta1']
    beta2 = config['beta2']
    epsilon = config['epsilon']

    var_index = 0
    for layer_vars, layer_grads in zip(variables, gradients):
        for p, g in zip(layer_vars, layer_grads):
            m = state['m'].setdefault(var_index, np.zeros_like(g))
            v = state['v'].setdefault(var_index, np.zeros_like(g))

            m *= beta1
            m += (1 - beta1) * g

            v *= beta2
            v += (1 - beta2) * (g ** 2)

            m_hat = m / (1 - beta1 ** state['t'])
            v_hat = v / (1 - beta2 ** state['t'])

            p -= lr * m_hat / (np.sqrt(v_hat) + epsilon)

            var_index += 1

    return variables, state

# Layers for advanced track homework
You **don't need** to implement it if you are working on `homework_main-basic.ipynb`

## 12. Conv2d [Advanced]
- input:   **`batch_size x in_channels x h x w`**
- output: **`batch_size x out_channels x h x w`**

You should implement something like pytorch `Conv2d` layer with `stride=1` and zero-padding outside of image using `scipy.signal.correlate` function.

Practical notes:
- While the layer name is "convolution", the most of neural network frameworks (including tensorflow and pytorch) implement operation that is called [correlation](https://en.wikipedia.org/wiki/Cross-correlation#Cross-correlation_of_deterministic_signals) in signal processing theory. So **don't use** `scipy.signal.convolve` since it implements [convolution](https://en.wikipedia.org/wiki/Convolution#Discrete_convolution) in terms of signal processing.
- It's rather ok to implement convolution over 4d array using 2 nested loops: one over batch size dimension and another one over output filters dimension
- Having troubles with understanding how to implement the layer? 
 - Check the last year video of lecture 3 (starting from ~1:14:20)
 - May the google be with you

In [38]:
import numpy as np
import scipy.signal


class Conv2d(Module):
    def __init__(self, in_channels, out_channels, kernel_size):
        super(Conv2d, self).__init__()
        assert kernel_size % 2 == 1, kernel_size

        stdv = 1. / np.sqrt(in_channels)
        self.W = np.random.uniform(-stdv, stdv, size=(out_channels,
                                   in_channels, kernel_size, kernel_size))
        self.b = np.random.uniform(-stdv, stdv, size=(out_channels,))
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size

        self.gradW = np.zeros_like(self.W)
        self.gradb = np.zeros_like(self.b)

    def updateOutput(self, input):
        pad_size = self.kernel_size // 2
        padded_input = np.pad(input, ((
            0, 0), (0, 0), (pad_size, pad_size), (pad_size, pad_size)), mode='constant')
        batch_size, _, h, w = input.shape
        self.output = np.zeros((batch_size, self.out_channels, h, w))

        for b in range(batch_size):
            for out_c in range(self.out_channels):
                conv_sum = np.zeros((h, w))
                for in_c in range(self.in_channels):
                    conv_sum += scipy.signal.correlate(
                        padded_input[b, in_c], self.W[out_c, in_c], mode='valid')
                self.output[b, out_c] = conv_sum + self.b[out_c]
        return self.output

    def updateGradInput(self, input, gradOutput):
        pad_size = self.kernel_size // 2
        batch_size, _, h, w = input.shape
        padded_gradOutput = np.pad(gradOutput, ((
            0, 0), (0, 0), (pad_size, pad_size), (pad_size, pad_size)), mode='constant')
        self.gradInput = np.zeros_like(input)

        for b in range(batch_size):
            for in_c in range(self.in_channels):
                grad_sum = np.zeros((h, w))
                for out_c in range(self.out_channels):
                    flipped_W = np.flip(self.W[out_c, in_c], (0, 1))
                    grad_sum += scipy.signal.correlate(
                        padded_gradOutput[b, out_c], flipped_W, mode='valid')
                self.gradInput[b, in_c] = grad_sum
        return self.gradInput

    def accGradParameters(self, input, gradOutput):
        pad_size = self.kernel_size // 2
        batch_size, _, h, w = input.shape
        padded_input = np.pad(input, ((
            0, 0), (0, 0), (pad_size, pad_size), (pad_size, pad_size)), mode='constant')

        for out_c in range(self.out_channels):
            for in_c in range(self.in_channels):
                grad_sum = np.zeros((self.kernel_size, self.kernel_size))
                for b in range(batch_size):
                    grad_sum += scipy.signal.correlate(
                        padded_input[b, in_c], gradOutput[b, out_c], mode='valid')
                self.gradW[out_c, in_c] += grad_sum

        self.gradb += np.sum(gradOutput, axis=(0, 2, 3))

    def zeroGradParameters(self):
        self.gradW.fill(0)
        self.gradb.fill(0)

    def getParameters(self):
        return [self.W, self.b]

    def getGradParameters(self):
        return [self.gradW, self.gradb]

    def __repr__(self):
        s = self.W.shape
        q = 'Conv2d %d -> %d' % (s[1], s[0])
        return q

## 13. MaxPool2d [Advanced]
- input:   **`batch_size x n_input_channels x h x w`**
- output: **`batch_size x n_output_channels x h // kern_size x w // kern_size`**

You are to implement simplified version of pytorch `MaxPool2d` layer with stride = kernel_size. Please note, that it's not a common case that stride = kernel_size: in AlexNet and ResNet kernel_size for max-pooling was set to 3, while stride was set to 2. We introduce this restriction to make implementation simplier.

Practical notes:
- During forward pass what you need to do is just to reshape the input tensor to `[n, c, h / kern_size, kern_size, w / kern_size, kern_size]`, swap two axes and take maximums over the last two dimensions. Reshape + axes swap is sometimes called space-to-batch transform.
- During backward pass you need to place the gradients in positions of maximal values taken during the forward pass
- In real frameworks the indices of maximums are stored in memory during the forward pass. It is cheaper than to keep the layer input in memory and recompute the maximums.

In [36]:
class MaxPool2d(Module):
    def __init__(self, kernel_size):
        super(MaxPool2d, self).__init__()
        self.kernel_size = kernel_size
        self.gradInput = None

    def updateOutput(self, input):
        batch_size, channels, input_h, input_w = input.shape
        k = self.kernel_size
        assert input_h % k == 0
        assert input_w % k == 0

        out_h = input_h // k
        out_w = input_w // k

        self.output = np.zeros(
            (batch_size, channels, out_h, out_w), dtype=input.dtype)
        self.max_indices = np.zeros_like(input, dtype=bool)

        for b in range(batch_size):
            for c in range(channels):
                for i in range(out_h):
                    for j in range(out_w):
                        h_start = i * k
                        w_start = j * k
                        window = input[b, c, h_start:h_start +
                                       k, w_start:w_start+k]
                        max_val = np.max(window)
                        self.output[b, c, i, j] = max_val
                        max_mask = (window == max_val)
                        self.max_indices[b, c, h_start:h_start +
                                         k, w_start:w_start+k] = max_mask
        return self.output

    def updateGradInput(self, input, gradOutput):
        batch_size, channels, input_h, input_w = input.shape
        k = self.kernel_size
        out_h = input_h // k
        out_w = input_w // k

        self.gradInput = np.zeros_like(input, dtype=gradOutput.dtype)

        for b in range(batch_size):
            for c in range(channels):
                for i in range(out_h):
                    for j in range(out_w):
                        h_start = i * k
                        w_start = j * k
                        max_mask = self.max_indices[b, c,
                                                    h_start:h_start+k, w_start:w_start+k]
                        num_max = np.sum(max_mask)
                        grad = gradOutput[b, c, i, j] / num_max
                        self.gradInput[b, c, h_start:h_start+k,
                                       w_start:w_start+k] += max_mask * grad
        return self.gradInput

    def __repr__(self):
        q = 'MaxPool2d, kern %d, stride %d' % (
            self.kernel_size, self.kernel_size)
        return q

### Flatten layer
Just reshapes inputs and gradients. It's usually used as proxy layer between Conv2d and Linear.

In [37]:
class Flatten(Module):
    def __init__(self):
        super(Flatten, self).__init__()

    def updateOutput(self, input):
        self.output = input.reshape(len(input), -1)
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput.reshape(input.shape)
        return self.gradInput

    def __repr__(self):
        return "Flatten"